# MCP 서버를 AgentCore Gateway에 통합하기

## 개요

수천 개의 개발 팀이 각각 여러 도구, 프롬프트, 리소스를 포함한 여러 MCP 서버를 운영하는 대규모 기업에서는 기능을 관리하고 검색하는 일이 매우 중요한 과제가 됩니다.

* **기능 검색 및 공유**: 팀은 조직 전체의 도구, 프롬프트, 리소스를 검색하고 사용하는 데 어려움을 겪습니다. 각 팀이 별도의 Gateway를 유지해야 할까요? 운영 부담을 늘리지 않으면서 Gateway URL을 공유하고 중앙 레지스트리를 관리하려면 어떻게 해야 할까요?
* **Gateway 관리**: MCP 서버마다 별도의 Gateway를 유지하는 방식은 규모가 커지면 빠르게 관리하기 어려워집니다.
* **인증 복잡성**: 여러 MCP 서버의 인증과 권한 부여를 관리하는 일은 점점 더 복잡해지며, 특히 민감한 엔터프라이즈 데이터를 다룰 때 더욱 그렇습니다.
* **유지 관리 부담**: MCP 사양 업데이트를 따라가려면 모든 구현에서 지속적인 재작업과 테스트가 필요합니다.

AgentCore Gateway는 기존 MCP 서버 구현을 대상으로 온보딩할 수 있으며, 세 가지 MCP 기본 유형인 **도구**, **프롬프트**, **리소스**(URI 템플릿 기반 리소스 포함)를 해당 대상으로 전달합니다. 이 튜토리얼에서는 AgentCore Runtime에 호스팅된 FastMCP 서버를 사용하여 이 개념을 처음부터 끝까지 살펴봅니다.

![구성도](./images/mcp-server-target.png)

## 워크숍 진행 순서

| 단계 | 수행 내용 |
|---|---|
| **1** | Notebook 환경(환경 변수, 유틸리티, 로깅)을 설정합니다. |
| **2** | Cognito 인바운드 인증, IAM 역할, AgentCore Gateway 순서로 생성합니다. |
| **3** | 도구, 프롬프트, 리소스, 템플릿 기반 리소스를 포함하는 FastMCP 서버를 AgentCore Runtime에 배포합니다. |
| **4** | MCP 서버를 Gateway 대상으로 연결합니다(아웃바운드 OAuth, 대상 생성, 확인, 인바운드 토큰). |
| **5** | MCP를 통해 도구를 호출하는 Strands 에이전트로 Gateway를 통해 **도구**를 사용합니다. |
| **6** | Gateway를 통해 **프롬프트**를 사용합니다: `prompts/list` 및 `prompts/get`. |
| **7** | Gateway를 통해 **리소스**를 사용합니다: `resources/list`, `resources/read`, `resources/templates/list`. |
| **8** | 리소스 URI가 겹치는 *두 번째* MCP 서버를 배포하고 `resourcePriority` 필드를 사용한 대상 간 충돌 해결을 실습합니다. |
| **9** | 리소스를 정리합니다. |

## 튜토리얼 세부 정보

| 정보                  | 세부 정보                                                                            |
|:---------------------|:-------------------------------------------------------------------------------------|
| 튜토리얼 유형         | 대화형                                                                                |
| AgentCore 구성 요소  | AgentCore Gateway, AgentCore Identity, AgentCore Runtime                             |
| 에이전틱 프레임워크   | Strands Agents                                                                       |
| Gateway 대상 유형    | MCP 서버                                                                             |
| MCP 기본 요소        | 도구, 프롬프트, 리소스(정적 및 템플릿 기반)                                          |
| 인바운드 인증 IdP    | Amazon Cognito(다른 서비스도 사용 가능)                                              |
| 아웃바운드 인증      | Amazon Cognito(다른 서비스도 사용 가능)                                              |
| LLM 모델             | Anthropic Claude Haiku 4.5                                                           |
| 튜토리얼 구성 요소   | AgentCore Gateway 생성, 도구/프롬프트/리소스 호출, resourcePriority 실습            |
| 튜토리얼 분야        | 여러 분야                                                                            |
| 예제 난이도          | 쉬움                                                                                 |
| 사용 SDK             | boto3                                                                                |

### 1단계: 설정 및 사전 요구 사항

이 튜토리얼을 실행하려면 다음 항목이 필요합니다.
* Jupyter notebook(Python 3.10 이상 커널)
* AgentCore CLI용 Node.js 및 npm(아래 셀에서 `@aws/agentcore`를 전역으로 설치)
* `aws configure`, 환경 변수 또는 인스턴스 역할을 통해 구성한 AWS 자격 증명 및 리전
* CloudFormation, Cognito IDP, IAM, Bedrock AgentCore(제어 및 런타임), `bedrock:InvokeModel`에 대한 IAM 권한(5단계의 Strands 에이전트에서 사용)
* `global.anthropic.claude-haiku-4-5-20251001-v1:0` Bedrock 모델에 대한 액세스 권한

In [ ]:
# 현재 디렉터리의 requirements 파일 또는 pyproject.toml 파일에서 설치
!pip install --force-reinstall -U -r requirements.txt --quiet

In [ ]:
!npm install -g @aws/agentcore

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# 유틸리티 가져오기
import utils
import logging
import boto3
import json

# Notebook 환경의 로깅 구성
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    handlers=[logging.StreamHandler()],
)

# 특정 로거 수준 설정
logging.getLogger("strands").setLevel(logging.INFO)

REGION = boto3.Session().region_name
COGNITO_STACK_NAME = "agentcore-gateway-lab"
TEMPLATE_PATH = "cloudformation/cognito-signup-stack.yaml"
MCP_SERVER_NAME = "lab1mcp"
GATEWAY_NAME = "ac-gateway-mcp-server"

cfn = boto3.client("cloudformation", region_name=REGION)
cognito = boto3.client("cognito-idp", region_name=REGION)

In [ ]:
REGION

### 2단계: AgentCore Gateway 생성

### 2.1단계: CloudFormation을 통해 Cognito 배포

[`cloudformation/cognito-signup-stack.yaml`](cloudformation/cognito-signup-stack.yaml)을 배포합니다.

참고: 이 실습에서는 AgentCore Gateway 패턴에 집중할 수 있도록 인바운드 인증에 Cognito를 사용하도록 AgentCore Gateway를 구성합니다. 엔터프라이즈 워크로드에서는 인바운드 인증에 OAuth 2.0 호환 자격 증명 공급자(예: Entra ID, Auth0, Okta)를 사용할 수 있습니다. 자세한 내용은 [자격 증명 공급자 설정](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/identity-idps.html)을 참조하세요. AgentCore Gateway와 대상 간의 아웃바운드 권한 부여에는 [AgentCore Gateway Identity 자격 증명 관리](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/what-is-bedrock-agentcore.html)를 설정하는 것이 좋습니다.

In [ ]:
outputs = utils.deploy_cognito_stack(cfn, COGNITO_STACK_NAME, TEMPLATE_PATH)

# Gateway 인바운드
gw_user_pool_id = outputs["UserPoolId"]
gw_client_id = outputs["GatewayClientId"]
gw_cognito_discovery_url = outputs["DiscoveryUrl"]
scopeString = outputs["GatewayScope"]
token_endpoint = outputs["TokenEndpoint"]
gw_client_secret = cognito.describe_user_pool_client(
    UserPoolId=gw_user_pool_id, ClientId=gw_client_id
)["UserPoolClient"]["ClientSecret"]

# MCP 서버로의 아웃바운드(동일한 풀)
runtime_user_pool_id = gw_user_pool_id
runtime_client_id = outputs["MCPClientId"]
runtime_cognito_discovery_url = gw_cognito_discovery_url
runtimeScopeString = outputs["MCPScope"]
runtime_client_secret = cognito.describe_user_pool_client(
    UserPoolId=runtime_user_pool_id, ClientId=runtime_client_id
)["UserPoolClient"]["ClientSecret"]

print(f"Stack:              {COGNITO_STACK_NAME}")
print(f"User Pool ID:       {gw_user_pool_id}")
print(f"Discovery URL:      {gw_cognito_discovery_url}")
print(f"Token endpoint:     {token_endpoint}")
print(f"Gateway client ID:  {gw_client_id}")
print(f"MCP client ID:      {runtime_client_id}")
print(f"Gateway scope:      {scopeString}")
print(f"MCP scope:          {runtimeScopeString}")

### 2.2단계: AgentCore Gateway IAM 역할 생성

In [ ]:
agentcore_gateway_iam_role = utils.create_agentcore_gateway_role_with_region(
    GATEWAY_NAME, REGION
)
print("Agentcore gateway role ARN: ", agentcore_gateway_iam_role["Role"]["Arn"])

### 2.3단계: AgentCore Gateway 생성

In [ ]:
# Cognito 권한 부여자를 사용하여 CreateGateway 호출. 이전 단계에서 생성한 Cognito 사용자 풀 사용
import boto3

gateway_client = boto3.client("bedrock-agentcore-control", region_name=REGION)
auth_config = {
    "customJWTAuthorizer": {
        "allowedClients": [
            gw_client_id
        ],  # 클라이언트는 Cognito에 구성된 ClientId와 반드시 일치해야 함. 예: 7rfbikfsm51j2fpaggacgng84g
        "discoveryUrl": gw_cognito_discovery_url,
    }
}
create_response = gateway_client.create_gateway(
    name=GATEWAY_NAME,
    roleArn=agentcore_gateway_iam_role["Role"][
        "Arn"
    ],  # IAM 역할에는 Gateway 생성/목록 조회/조회/삭제 권한이 있어야 함
    protocolType="MCP",
    protocolConfiguration={
        "mcp": {"supportedVersions": ["2025-11-25"], "searchType": "SEMANTIC"}
    },
    authorizerType="CUSTOM_JWT",
    authorizerConfiguration=auth_config,
    description="AgentCore Gateway with MCP Server target",
)
print(create_response)
# GatewayTarget 생성에 사용할 GatewayID 검색
gatewayID = create_response["gatewayId"]
gatewayURL = create_response["gatewayUrl"]
print(gatewayID)

### 3단계: AgentCore Runtime에 MCP 서버 배포

In [ ]:
!cd mcpservers && agentcore add agent \
    --name {MCP_SERVER_NAME} \
    --type byo \
    --language Python \
    --protocol MCP \
    --code-location app/labmcp \
    --authorizer-type CUSTOM_JWT \
    --discovery-url {runtime_cognito_discovery_url} \
    --allowed-clients {runtime_client_id} \
    --allowed-scopes {runtimeScopeString}

### 3.2단계: MCP 서버 코드

In [ ]:
from IPython.display import Code

Code("mcpservers/app/labmcp/main.py", language="python")

### 3.3단계: AgentCore CLI를 통해 배포

In [ ]:
!cd mcpservers && agentcore deploy

In [ ]:
agent = utils.get_agent_status(MCP_SERVER_NAME)

mcp_arn = agent["identifier"]
mcp_url = agent["invocationUrl"]
mcp_id = mcp_arn.split("/")[-1]

print(f"mcp_arn: {mcp_arn}")
print(f"mcp_id:  {mcp_id}")
print(f"mcp_url: {mcp_url}")

### 4단계: MCP 서버를 Gateway 대상으로 연결

### 4.1단계: 아웃바운드 인증 구성(OAuth2 자격 증명 공급자)

Gateway가 AgentCore Runtime의 MCP 서버를 호출할 때 아웃바운드 인증에 사용할 AgentCore Identity Resource Credential Provider를 생성합니다.

In [ ]:
import boto3

identity_client = boto3.client("bedrock-agentcore-control", region_name=REGION)

cognito_provider = identity_client.create_oauth2_credential_provider(
    name=f"{GATEWAY_NAME}-identity",
    credentialProviderVendor="CustomOauth2",
    oauth2ProviderConfigInput={
        "customOauth2ProviderConfig": {
            "oauthDiscovery": {
                "discoveryUrl": runtime_cognito_discovery_url,
            },
            "clientId": runtime_client_id,
            "clientSecret": runtime_client_secret,
        }
    },
)
cognito_provider_arn = cognito_provider["credentialProviderArn"]
print(cognito_provider_arn)

### 4.2단계: Gateway 대상 생성

In [ ]:
create_gateway_target_response = gateway_client.create_gateway_target(
    name="mcp-server-target",
    gatewayIdentifier=gatewayID,
    targetConfiguration={
        "mcp": {
            "mcpServer": {
                "endpoint": mcp_url,
                "resourcePriority": 10,  # 리소스 URI 충돌 시 낮은 값이 우선하며 8단계에서 사용됨
            }
        }
    },
    credentialProviderConfigurations=[
        {
            "credentialProviderType": "OAUTH",
            "credentialProvider": {
                "oauthCredentialProvider": {
                    "providerArn": cognito_provider_arn,
                    "scopes": [runtimeScopeString],
                }
            },
        },
    ],
    # 클라이언트가 제공한 `Mcp-Session-Id`를 양방향으로 런타임에 전달
    # AgentCore Runtime은 이를 사용하여 요청을 특정 microvm에 고정하므로
    # 이후 동일한 id를 사용하는 호출은 같은 인스턴스를 재사용함
    metadataConfiguration={
        "allowedRequestHeaders": ["Mcp-Session-Id"],
        "allowedResponseHeaders": ["Mcp-Session-Id"],
    },
)
gatewayTargetID = create_gateway_target_response["targetId"]
print(f"Created target: {gatewayTargetID}")

### 4.3단계: Gateway 대상이 READY 상태인지 확인

In [ ]:
list_targets_response = gateway_client.list_gateway_targets(gatewayIdentifier=gatewayID)
print(list_targets_response)

### 4.4단계: 인바운드 액세스 토큰 가져오기

In [ ]:
print(
    "Requesting the access token from Amazon Cognito authorizer...May fail for some time till the domain name propagation completes"
)
token_response = utils.get_token(
    token_endpoint, gw_client_id, gw_client_secret, scopeString
)
token = token_response["access_token"]
print("Token response:", token)

## 5단계: Gateway를 통해 도구 사용

Gateway는 표준 MCP `tools/list` 및 `tools/call` 호출을 통해 MCP 서버의 도구를 제공합니다. 아래에서는 먼저 캐시 방식과 실시간 방식의 동작을 살펴본 다음, Gateway를 통해 도구를 호출하는 Strands 에이전트를 실행합니다.

### 5.1단계: Gateway를 통한 도구의 처리 흐름

**ListTools (`tools/list`).** Gateway는 캐시 우선 방식에 따라 MCP 대상에서 이전에 동기화한 도구 정의에 대한 액세스를 제공합니다. 클라이언트가 `tools/list`를 호출하면 Gateway는 MCP 서버를 실시간으로 호출하는 대신 캐시된 정규화 도구 정의를 반환합니다. 캐시는 대상 생성/업데이트 중 **암시적 동기화**되거나 `SynchronizeGatewayTargets`를 통해 **명시적 동기화**됩니다. 자세한 내용은 다음 Notebook인 `02-mcp-target-synchronization.ipynb`를 참조하세요. 여기에는 `listingMode='DYNAMIC'`을 사용해 `tools/list`를 MCP 서버에 실시간으로 전달하는 방법도 포함됩니다.

**InvokeTool (`tools/call`).** 도구 호출은 *항상* 실시간으로 처리됩니다. Gateway는 MCP 서버와 세션을 열고, 필요한 경우 AgentCore Identity에서 새로운 아웃바운드 자격 증명을 가져온 후 호출을 전달합니다. 클라이언트가 전송하는 도구 이름에는 대상 접두사 `{targetName}___{toolName}`(밑줄 3개)가 포함되어야 합니다.

![목록 조회](images/mcp-server-list-tools.png)

![호출](images/mcp-server-invoke-tool.png)

AgentCore Gateway 검색 기능에 관한 더 많은 예제는 [03 - 도구 검색](../03-search-tools)을 참조하세요.

### 5.2단계: Strands 에이전트가 Gateway를 통해 도구 호출

In [ ]:
from strands.models import BedrockModel
from mcp.client.streamable_http import streamablehttp_client
from strands.tools.mcp.mcp_client import MCPClient
from strands import Agent


def get_token():
    token = utils.get_token(token_endpoint, gw_client_id, gw_client_secret, scopeString)
    return token["access_token"]


def create_streamable_http_transport():
    return streamablehttp_client(
        gatewayURL, headers={"Authorization": f"Bearer {get_token()}"}
    )


client = MCPClient(create_streamable_http_transport)

## ~/.aws/credentials에 구성된 IAM 그룹/사용자에는 Bedrock 모델에 대한 액세스 권한이 있어야 함
yourmodel = BedrockModel(
    model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0",  # 리전에 따라 model_id를 업데이트해야 할 수 있음
    temperature=0.7,
    max_tokens=500,  # 응답 길이 제한
)

with client:
    # listTools 호출
    tools = client.list_tools_sync()
    # 모델과 도구를 사용하여 Agent 생성
    agent = Agent(
        model=yourmodel, tools=tools
    )  ## 원하는 모델로 교체 가능
    # 샘플 프롬프트로 에이전트 호출. MCP listTools만 호출하여 LLM이 액세스할 수 있는 도구 목록을 가져오며, 아래에서는 실제로 도구를 호출하지 않음
    agent("Hi, can you list all tools available to you")
    # 간소화된 프롬프트 및 오류 처리
    result = agent("Update order 123")

### 5.3단계: MCP 사양의 도구 `title` 및 `annotations` 지원

MCP 사양(2025-06-18 이후)에서는 도구가 다음 정보를 제공할 수 있습니다.

- **`title`** - 함수/도구의 `name`과 구분되는, 사람이 읽기 쉬운 표시 이름입니다. 사용자에게 도구 카탈로그를 보여 주는 클라이언트(예: 호출할 도구를 선택하는 LLM 기반 에이전트)는 표시에 `title`을 우선 사용해야 합니다.
- **`annotations`** - 도구 호출 전에 클라이언트가 안전성 및 멱등성을 판단하는 데 사용할 수 있는 구조화된 힌트 객체입니다.
  - `title` - 최상위 `title`과 동일합니다(주석에 반복 표시).
  - `readOnlyHint` - 도구가 환경 상태를 변경하지 않습니다.
  - `destructiveHint` - 도구가 데이터를 삭제하거나 파기할 수 있습니다.
  - `idempotentHint` - 동일한 인수로 다시 호출해도 추가 효과가 없습니다.
  - `openWorldHint` - 도구가 서버 도메인 외부의 리소스에 액세스할 수 있습니다(예: 외부 HTTP 요청 수행).

In [ ]:
import uuid
from gateway_mcp_client import GatewayMCPClient


def _get_inbound_token() -> str:
    return utils.get_token(token_endpoint, gw_client_id, gw_client_secret, scopeString)[
        "access_token"
    ]


session_id = str(uuid.uuid4())

In [ ]:
mcp = GatewayMCPClient(gatewayURL, _get_inbound_token, session_id=session_id)

mcp.list_tools()

## 6단계: Gateway를 통해 프롬프트 사용

프롬프트는 MCP 서버가 AI 생성을 위해 제공하는 매개변수화된 메시지 템플릿입니다. Gateway는 다음 두 가지 MCP 메서드를 전달합니다.

- `prompts/list` - 대상이 기본값인 `listingMode='DEFAULT'`를 사용하면 Gateway의 카탈로그(동기화 중 캐시됨)에서 제공됩니다. 프롬프트 이름은 대상 접두사 `{targetName}___{promptName}`(도구와 동일하게 밑줄 3개)와 함께 반환됩니다.
- `prompts/get` - `listingMode`와 관계없이 *항상* 다운스트림 MCP 서버로 실시간 프록시됩니다. 프롬프트 `name` 인수에는 반드시 `targetName___` 접두사가 포함되어야 합니다.

### 6.1단계: `prompts/list` - Gateway를 통해 사용할 수 있는 프롬프트 목록 조회

In [ ]:
mcp = GatewayMCPClient(gatewayURL, _get_inbound_token, session_id=session_id)


print(json.dumps(mcp.list_prompts(), indent=2))

### 6.2단계: `prompts/get` - 렌더링된 프롬프트 가져오기

In [ ]:
mcp = GatewayMCPClient(gatewayURL, _get_inbound_token, session_id=session_id)

result = mcp.get_prompt(
    name="mcp-server-target___order_summary_prompt",
    arguments={"orderId": "123"},
)
print(json.dumps(result, indent=2))

## 7단계: Gateway를 통해 리소스 사용

리소스는 MCP 서버가 URI를 통해 제공하는 주소 지정 가능한 콘텐츠입니다. 템플릿 기반 리소스는 [RFC 6570](https://datatracker.ietf.org/doc/html/rfc6570) URI 템플릿을 사용하므로 하나의 핸들러가 여러 구체적인 URI를 처리할 수 있습니다. Gateway는 다음 세 가지 MCP 메서드를 전달합니다.

- `resources/list` 및 `resources/templates/list` - `listingMode='DEFAULT'`에서는 Gateway 카탈로그에서 제공되고, `listingMode='DYNAMIC'`에서는 실시간으로 전달됩니다. 리소스 URI는 **원문 그대로** 반환됩니다. `target___` 접두사가 없으며 MCP 서버의 원래 URI가 변경 없이 전달됩니다.
- `resources/read` - `listingMode`와 관계없이 *항상* 다운스트림 MCP 서버로 실시간 프록시됩니다.

### 7.1단계: `resources/list` 및 `resources/read`

> **보안 경고**
>
> 리소스 URI는 다운스트림 MCP 서버 대상에서 제공하며 Gateway에서는 이를 검증하거나 정제하지 않습니다. 악의적이거나 침해된 MCP 서버가 내부 엔드포인트(SSRF) 또는 로컬 파일 시스템 경로(예: `file:///etc/passwd`)를 가리키는 URI를 반환할 수 있습니다. 리소스 URI를 사용하기 전에 검증하고 정제하며, 신뢰할 수 없는 MCP 서버 대상의 URI를 자동으로 가져오거나 렌더링하지 마세요.

In [ ]:
mcp = GatewayMCPClient(gatewayURL, _get_inbound_token, session_id=session_id)

print("--- resources/list ---")
print(json.dumps(mcp.list_resources(), indent=2))

print("\n--- resources/read orders://catalog ---")
print(json.dumps(mcp.read_resource("orders://catalog"), indent=2))

### 7.2단계: `resources/templates/list` 및 템플릿 기반 URI 읽기

리소스 템플릿은 [RFC 6570 URI 템플릿](https://datatracker.ietf.org/doc/html/rfc6570)(예: `orders://{orderId}/details`)을 사용합니다. 클라이언트가 매개변수를 채워 구체적인 URI를 생성하면 표준 MCP 서버가 `resources/read`를 통해 이를 제공합니다.

아래의 첫 번째 호출인 `resources/templates/list`는 예상대로 작동하며 업스트림 서버가 등록한 템플릿을 반환합니다.

두 번째 호출은 `resources/read`이며, 해당 템플릿에서 파생된 구체적인 URI `orders://123/details`를 대상으로 실행합니다.

In [ ]:
mcp = GatewayMCPClient(gatewayURL, _get_inbound_token, session_id=session_id)

print("--- resources/templates/list ---")
print(json.dumps(mcp.list_resource_templates(), indent=2))

print("\n--- resources/read orders://123/details ---")
print(json.dumps(mcp.read_resource("orders://123/details"), indent=2))

## 8단계: `resourcePriority`를 사용한 여러 대상 간 충돌 해결

도구와 프롬프트에는 `{targetName}___{name}` 네임스페이스가 자동으로 지정되므로 대상 간 충돌이 발생하지 않습니다. 리소스는 이와 달리 Gateway가 리소스 URI를 원문 그대로 반환합니다. 두 대상이 동일한 URI를 제공하면 Gateway는 `resourcePriority`가 가장 낮은 대상으로 읽기 요청을 라우팅하여 충돌을 해결합니다(범위 `0–1000`, 기본값 `1000`, 낮은 값 우선).

이 단계에서는 별도의 AgentCore Runtime을 시작하지 않고 공개 **Exa MCP 서버**(`https://mcp.exa.ai/mcp`)를 `resourcePriority=100`인 *두 번째* Gateway 대상으로 추가합니다. 그런 다음 Exa의 리소스 목록에서 하나를 선택하고 [`mcpservers/app/labmcp/main.py`](mcpservers/app/labmcp/main.py)에 동일한 URI를 추가하여 해당 리소스를 **섀도잉**합니다. 이 대상은 4.2단계에서 이미 `resourcePriority=10`으로 설정했습니다. 낮은 값이 우선하므로 해당 URI에는 런타임 서버의 콘텐츠가 Exa의 콘텐츠보다 우선 적용됩니다.

### 8.1단계: Exa MCP 서버를 `resourcePriority=100`인 두 번째 Gateway 대상으로 추가

`https://mcp.exa.ai/mcp`는 공개 관리형 MCP 서버(Exa 검색)입니다. Gateway는 자체 런타임뿐만 아니라 서드 파티 MCP 서비스도 지원하므로 이 서버를 대상에 바로 연결할 수 있습니다. Exa는 런타임 서버가 이미 섀도잉하고 있는 `exa://tools/list` 리소스 하나를 제공합니다([`mcpservers/app/labmcp/main.py`](mcpservers/app/labmcp/main.py) 참조). Exa를 연결하면 두 대상이 모두 해당 URI를 제공하며, `resourcePriority`에 따라 읽기 요청에 응답할 대상이 결정됩니다.

Exa의 `resources/list` 및 `resources/read`에는 인증 없이 액세스할 수 있으므로 이 데모는 API 키 없이 실행됩니다. Exa의 *도구*(`web_search_exa`, `web_fetch_exa` 등)를 사용하려면 `EXA_API_KEY`를 설정하고 엔드포인트 URL의 `exaApiKey` 쿼리 매개변수로 전달해야 합니다. 다음 셀에서는 이를 조건부로 수행합니다. 프로덕션 환경에서는 키를 URL에 직접 포함하지 말고 AgentCore Identity에서 **API 키 자격 증명 공급자**를 설정하는 것이 좋습니다.

In [ ]:
create_exa_target_response = gateway_client.create_gateway_target(
    name="mcp-server-target-exa",
    gatewayIdentifier=gatewayID,
    targetConfiguration={
        "mcp": {
            "mcpServer": {
                "endpoint": "https://mcp.exa.ai/mcp",
                "resourcePriority": 100,  # 런타임의 10보다 높으므로 충돌 시 런타임이 우선함
            }
        }
    },
)
exaTargetID = create_exa_target_response["targetId"]
print(f"Created Exa target: {exaTargetID}")

### 8.2단계: 병합된 리소스 카탈로그 확인

Gateway를 통해 `resources/list`를 호출합니다. 이제 응답에는 런타임 서버의 `orders://...`와 Exa의 `exa://tools/list` 등 **두** 대상의 리소스가 포함됩니다. 두 대상이 모두 `exa://tools/list`를 제공하므로 이 항목이 **두 번** 표시됩니다.

In [ ]:
mcp = GatewayMCPClient(gatewayURL, _get_inbound_token, session_id=session_id)

print(json.dumps(mcp.list_resources(), indent=2))

### 8.3단계: 충돌하는 URI 읽기 - 런타임 우선

두 대상은 서로 다른 콘텐츠로 `exa://tools/list`를 제공합니다. 런타임은 [`mcpservers/app/labmcp/main.py`](mcpservers/app/labmcp/main.py)의 고정 문자열을 반환하고, Exa는 실제 도구 카탈로그(web_search_exa, web_fetch_exa 등)를 반환합니다. 런타임 대상은 `resourcePriority=10`으로 생성되었습니다.
> "여러 대상이 동일한 리소스 URI를 제공하면 Gateway는 `resourcePriority` 값이 가장 낮은 대상으로 요청을 라우팅합니다."

In [ ]:
mcp = GatewayMCPClient(gatewayURL, _get_inbound_token, session_id=session_id)

print(json.dumps(mcp.read_resource("exa://tools/list"), indent=2))

### 8.4단계: 이름 지정 및 충돌 해결 요약

| 기능 | 대상 간 이름 지정 | 충돌 해결 |
|---|---|---|
| 도구 | `targetName___toolName`(밑줄 3개) | 이름에 네임스페이스가 지정되므로 충돌하지 않음 |
| 프롬프트 | `targetName___promptName`(밑줄 3개) | 이름에 네임스페이스가 지정되므로 충돌하지 않음 |
| 리소스 | 접두사 없이 URI를 원문 그대로 반환 | 대상의 `resourcePriority`(낮은 값 우선, 기본값 1000) |
| 리소스 템플릿 | 접두사 없이 URI 템플릿을 원문 그대로 반환 | `resourcePriority`를 따름 |

## 9단계: 리소스 정리

IAM 역할, IAM 정책, 자격 증명 공급자, AWS Lambda 함수, Cognito 사용자 풀, s3 버킷과 같은 추가 리소스도 생성되며 정리 과정에서 수동으로 삭제해야 할 수 있습니다. 이는 실행한 예제에 따라 달라집니다.

> **참고:** 다음 Notebook인 [02 - MCP 대상 동기화](02-mcp-target-synchronization.ipynb)를 진행하려면 다음 튜토리얼을 위해 이 리소스를 정리하는 것이 좋습니다.

In [ ]:
## 9.1단계: Gateway 삭제(Exa 대상을 포함한 두 대상을 연쇄적으로 삭제)
utils.delete_gateway(gateway_client, gatewayID)

In [ ]:
## 9.2단계: OAuth2 자격 증명 공급자 삭제
identity_client.delete_oauth2_credential_provider(name=f"{GATEWAY_NAME}-identity")

In [ ]:
## 9.3단계: AgentCore Runtime의 MCP 서버 삭제
!cd mcpservers && agentcore remove agent --name {MCP_SERVER_NAME} -y                                                                             
!cd mcpservers && agentcore deploy -y

In [ ]:
# ## 9.4단계: Cognito CloudFormation 스택 삭제(사용자 풀, 도메인, 리소스 서버, 모든 클라이언트)
# ## 다른 실습에서 이 스택을 사용하지 않을 때 Cognito 스택 삭제
print(f"Deleting stack {COGNITO_STACK_NAME}...")
cfn.delete_stack(StackName=COGNITO_STACK_NAME)
cfn.get_waiter("stack_delete_complete").wait(StackName=COGNITO_STACK_NAME)
print(f"✅ Stack {COGNITO_STACK_NAME} deleted")

In [ ]:
## 9.5단계: Gateway IAM 역할 삭제(CFN 스택에 포함되지 않음)
utils.delete_iam_role(f"agentcore-{GATEWAY_NAME}-role")